Now we will look at an implementation of the BPE (Byte-Pair Encoding) algorithm.
This is not an optimized version, meaning it is not the best version for actually running on a large corpus. The main purpose here is to understand the algorithm well by looking at the code.


## 1) First, We Need a Corpus


In [1]:
corpus = [
    "This is the Building-LLMs-from-scratch-In-Bangla Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",

]

## 2) Next, We Need to Pre-tokenize the Corpus
Since we want to build a BPE tokenizer like GPT-2, the GPT-2 tokenizer is used for pre-tokenization.


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

## 3) Counting How Many Times Each Word Appears

In [11]:
from collections import defaultdict

word_freqs= defaultdict(int)

for text in corpus:
    word_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word,offset in word_with_offsets]
    for word in new_words:
        word_freqs[word]+=1

print(word_freqs)


defaultdict(<class 'int'>, {'This': 3, 'Ġis': 2, 'Ġthe': 1, 'ĠBuilding': 1, '-': 5, 'LLMs': 1, 'from': 1, 'scratch': 1, 'In': 1, 'Bangla': 1, 'ĠCourse': 1, '.': 4, 'Ġchapter': 1, 'Ġabout': 1, 'Ġtokenization': 1, 'Ġsection': 1, 'Ġshows': 1, 'Ġseveral': 1, 'Ġtokenizer': 1, 'Ġalgorithms': 1, 'Hopefully': 1, ',': 1, 'Ġyou': 1, 'Ġwill': 1, 'Ġbe': 1, 'Ġable': 1, 'Ġto': 1, 'Ġunderstand': 1, 'Ġhow': 1, 'Ġthey': 1, 'Ġare': 1, 'Ġtrained': 1, 'Ġand': 1, 'Ġgenerate': 1, 'Ġtokens': 1})


## 4) Creating the Base Vocabulary
Now all the characters used in the corpus will be collected to create the base vocabulary.

In [13]:
alphabet = []
for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)

alphabet.sort()
print(alphabet)

[',', '-', '.', 'B', 'C', 'H', 'I', 'L', 'M', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ']


## 5) Adding the Special Token to the Vocabulary

The GPT-2 model uses a special token:
"<|endoftext|>"


So it is being added at the beginning of the vocabulary.


In [15]:
vocab = ["<|endoftext|>"]+ alphabet.copy()

Now the vocabulary will start with the special token, followed by all the characters from the alphabet.


In [17]:
vocab

['<|endoftext|>',
 ',',
 '-',
 '.',
 'B',
 'C',
 'H',
 'I',
 'L',
 'M',
 'T',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'y',
 'z',
 'Ġ']

## 6) Splitting Each Word into Characters

To start BPE training, each word must first be broken down at the character level.

In [18]:
splits ={ word:[c for c in word] for word in word_freqs.keys()} 

In [19]:
splits

{'This': ['T', 'h', 'i', 's'],
 'Ġis': ['Ġ', 'i', 's'],
 'Ġthe': ['Ġ', 't', 'h', 'e'],
 'ĠBuilding': ['Ġ', 'B', 'u', 'i', 'l', 'd', 'i', 'n', 'g'],
 '-': ['-'],
 'LLMs': ['L', 'L', 'M', 's'],
 'from': ['f', 'r', 'o', 'm'],
 'scratch': ['s', 'c', 'r', 'a', 't', 'c', 'h'],
 'In': ['I', 'n'],
 'Bangla': ['B', 'a', 'n', 'g', 'l', 'a'],
 'ĠCourse': ['Ġ', 'C', 'o', 'u', 'r', 's', 'e'],
 '.': ['.'],
 'Ġchapter': ['Ġ', 'c', 'h', 'a', 'p', 't', 'e', 'r'],
 'Ġabout': ['Ġ', 'a', 'b', 'o', 'u', 't'],
 'Ġtokenization': ['Ġ',
  't',
  'o',
  'k',
  'e',
  'n',
  'i',
  'z',
  'a',
  't',
  'i',
  'o',
  'n'],
 'Ġsection': ['Ġ', 's', 'e', 'c', 't', 'i', 'o', 'n'],
 'Ġshows': ['Ġ', 's', 'h', 'o', 'w', 's'],
 'Ġseveral': ['Ġ', 's', 'e', 'v', 'e', 'r', 'a', 'l'],
 'Ġtokenizer': ['Ġ', 't', 'o', 'k', 'e', 'n', 'i', 'z', 'e', 'r'],
 'Ġalgorithms': ['Ġ', 'a', 'l', 'g', 'o', 'r', 'i', 't', 'h', 'm', 's'],
 'Hopefully': ['H', 'o', 'p', 'e', 'f', 'u', 'l', 'l', 'y'],
 ',': [','],
 'Ġyou': ['Ġ', 'y', 'o', 'u'],

## 7) Function for Calculating Pair Frequency
Now, an important step in BPE training is:
finding which two consecutive characters/tokens appeared the most times.


In [20]:
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs


What is happening here?
- The split list of each word is being taken
- Pairs of consecutive tokens are being created
- The more times a word appears in the corpus, the more its pair frequency increases


## 8) Seeing Some Pair Frequencies


In [21]:
pair_freqs = compute_pair_freqs(splits)

for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break


('T', 'h'): 3
('h', 'i'): 3
('i', 's'): 5
('Ġ', 'i'): 2
('Ġ', 't'): 7
('t', 'h'): 3


## 9) Finding the Most Frequent Pair


In [22]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)


('Ġ', 't') 7


## 10) Learning the First Merge Rule

So the first merge rule will be:
('Ġ', 't') -> 'Ġt'

This is being stored as:


In [24]:
merges = {("Ġ", "t"): "Ġt"}
vocab.append("Ġt")


Meaning:
Now the new token "Ġt" has been added to the vocabulary.

## 11) Function for Applying the Merge
Now the new merge rule has to be applied to the splits of all words.


In [25]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits


Simply put:
This function does the following:
- looks at the token list of each word
- if a and b are found side by side
- then it merges them and makes a+b


## 12) Seeing the Result of the First Merge

In [26]:
splits = merge_pair("Ġ", "t", splits)
print(splits["Ġtrained"])



['Ġt', 'r', 'a', 'i', 'n', 'e', 'd']


## 13) Repeatedly Learning Merges with a Loop
Now this process will continue repeatedly until the vocabulary reaches the desired size.


In [27]:
vocab_size = 50

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])


What is the loop doing here?
Each time it:
- calculates pair frequency
- finds the most frequent pair
- merges the pair
- saves the merge rule
- adds the new token to the vocabulary


## 14) How Many Merge Rules Were Learned?

In [30]:
print(merges)

{('Ġ', 't'): 'Ġt', ('i', 's'): 'is', ('e', 'r'): 'er', ('Ġ', 'a'): 'Ġa', ('Ġt', 'o'): 'Ġto', ('e', 'n'): 'en', ('T', 'h'): 'Th', ('Th', 'is'): 'This', ('a', 't'): 'at', ('o', 'u'): 'ou', ('s', 'e'): 'se', ('Ġto', 'k'): 'Ġtok', ('Ġtok', 'en'): 'Ġtoken', ('n', 'd'): 'nd', ('Ġ', 'is'): 'Ġis'}


## 15) What Does the Final Vocabulary Look Like?


In [31]:
print(vocab)



['<|endoftext|>', ',', '-', '.', 'B', 'C', 'H', 'I', 'L', 'M', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ', 'Ġt', 'is', 'er', 'Ġa', 'Ġto', 'en', 'Th', 'This', 'at', 'ou', 'se', 'Ġtok', 'Ġtoken', 'nd', 'Ġis']


Meaning:

The final vocabulary is made up of:
- the special token
- the initial alphabet
- all the learned merged tokens


## 16) Important Note
If train_new_from_iterator() is used, the exact same vocabulary may not be produced.
Why?

Because if multiple pairs have the same frequency:
- our code takes the first one it finds
- but the Hugging Face Tokenizers library chooses the pair based on internal IDs
- That means even if the algorithm is the same, implementation details can differ.


## 17) Tokenizing New Text
Now new text can be tokenized using the learned merge rules.


In [32]:
def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    splits = [[l for l in word] for word in pre_tokenized_text]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])


What does this function do?
When new text comes:
- it pre-tokenizes first
- it splits each word into characters
- it applies the learned merge rules one by one
- finally, it returns the final token list


## 18) Example: Tokenizing

In [33]:
tokenize("This is not a token.")

['This', 'Ġis', 'Ġ', 'n', 'o', 't', 'Ġa', 'Ġtoken', '.']